# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Adapted CPU notebook (ORA ground-truth skeleton validation, phasepack+skimage) executed 10/11 cells on fresh CPU runtimes twice; each run hit a one-line variable-shadowing bug (pc then bg). Both fixed in source, but the 3-session Colab budget was exhausted before a clean fresh full run; notebook source no longer matches any executed output, so no verified publication. Resume with one fresh CPU session to validate the fixed notebook.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Quantitative analysis of plant ER architecture and dynamics.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-6f75092e4f4825ddd8125285022cfe18)


# Quantitative analysis of plant ER architecture and dynamics — AnalyzER segmentation vs ground truth (adapted Colab demo)

Reproduction of the skeleton-extraction validation subsection of:
**Stephens DJ et al. / Fricker MD et al., "Quantitative analysis of plant ER architecture and dynamics", *Nature Communications* 10:984 (2019).** doi:[10.1038/s41467-019-08893-9](https://doi.org/10.1038/s41467-019-08893-9) (PMCID PMC6395764)

**Scope (partial demonstration).** The authors distribute their AnalyzER MATLAB app / Windows executable and test data through the [Oxford Research Archive (ORA)](https://ora.ox.ac.uk/objects/uuid:cb0e2845-2a9c-495a-84f0-4dd2c5164463) (free-to-read, ODC-BY). The app itself cannot run in Colab, so this notebook **re-implements the documented tubule-skeleton pipeline in Python** and validates it exactly as the paper does: against a **manually digitized ground-truth skeleton** distributed in the ORA *"Processing example with ground truth"* package, using tolerance-based true-positive/false-negative/false-positive counting (AnalyzER manual §6.14–6.15). The paper reports Dice F1 = 0.94 ± 0.019 for its full GUI pipeline; this faithful-but-simplified port is expected to score lower (see the result and the interpretation cell).

**AI-assisted adaptation notice (EN):** This Python pipeline is an AI-assisted re-implementation from the AnalyzER v1 manual and the author's own per-example parameter file; the authors did not provide this Colab implementation. Parameters are taken from the authors' distributed `*_param.mat`; the executed example and checks were validated, but untested behavior is not guaranteed.

**AI支援による移植の注記 (JA):** このPythonパイプラインはAnalyzER v1マニュアルと著者配布のパラメータファイルに基づくAI支援の再実装です。著者自身のColab実装ではありません。実行例と検証は行いましたが、未検証の挙動は保証されません。

**Runtime: CPU only** — a single 1000×1000 frame with FFT-based phase congruency; no GPU operations exist in this method. Select **Runtime > Change runtime type > CPU** (or leave default), then **Runtime > Run all**. Expected end-to-end: ~2–4 min, ~25 MB of downloads.

## 1. Environment setup

Install the three non-preinstalled packages: `phasepack` (Peter Kovesi's own Python port of the phase-congruency algorithms that AnalyzER uses, PyPI v1.5), `aicspylibczi` (reader for the Zeiss `.czi` microscope format), and `h5py` (the authors' parameter files are MATLAB v7.3 HDF5). Japanese: 推論は不要でCPUのみ。必要なパッケージをインストールします。

In [ ]:
%pip -q install phasepack aicspylibczi h5py
import numpy as np, scipy, skimage, matplotlib
print('numpy', np.__version__, '| scipy', scipy.__version__,
      '| scikit-image', skimage.__version__, '| matplotlib', matplotlib.__version__)

## 2. Input acquisition — ORA "Processing example with ground truth"

Download the exact author-distributed test package (16.7 MB zip) from the Oxford Research Archive record for *AnalyzER v1* (file `ma30d05117261e506b4166aa58447cd8a`). This is dataset material, so it is downloaded **inside Colab only**. ORA rejects the default Python user agent, so we send an identifying one. We verify the ZIP magic bytes and record the SHA-256 for provenance. Japanese: 著者配布のテストデータ(zip 16.7 MB)をORAからColab内で取得し、検証します。

In [ ]:
import urllib.request, hashlib, zipfile, os
ORA_URL = ("https://ora.ox.ac.uk/objects/uuid:cb0e2845-2a9c-495a-84f0-4dd2c5164463"
           "/files/ma30d05117261e506b4166aa58447cd8a")  # "Processing example with ground truth.zip"
os.makedirs('/content/analyzer', exist_ok=True)
zp = '/content/analyzer/ground_truth_example.zip'
req = urllib.request.Request(ORA_URL, headers={'User-Agent': 'AnalyzER-colab-repro/1.0'})
data = urllib.request.urlopen(req, timeout=300).read()
assert data[:4] == b'PK\x03\x04', 'not a zip'
open(zp, 'wb').write(data)
print('bytes:', len(data), '| sha256:', hashlib.sha256(data).hexdigest())

## 3. Package contents

List the archive: it contains two Zeiss `.czi` confocal stacks of *Arabidopsis* leaf epidermis expressing GFP-HDEL, a manually digitized ground-truth skeleton `GFP-HDEL mv 50_Airyscan GT.tif` (1000×1000, network pixels = 0, background = 255), and the authors' MATLAB parameter files documenting the exact processing settings. Japanese: アーカイブ内容を一覧します。GT.tif は手作業で電子化した骨格、param.mat は著者の処理パラメータです。

In [ ]:
import zipfile
zf = zipfile.ZipFile(zp)
for i in zf.infolist():
    print(f'{i.file_size:>12,}  {i.filename}')
zf.extractall('/content/analyzer/extracted')

## 4. Read the authors' parameter file

The MATLAB v7.3 parameter file records the exact settings used for this example (read with h5py; MATLAB stores char arrays as uint16 codes). We extract the phase-congruency, background-subtraction and skeleton-threshold values and the pixel size — these parameterize the re-implementation below and its unit conversions. Japanese: 著者のパラメータファイルから位相一致度・背景除去・骨格化のしきい値と画素サイズを読み出します。

In [ ]:
import h5py

def chars(x):
    a = np.asarray(x).ravel()
    return ''.join(chr(int(v)) for v in a)

with h5py.File('/content/analyzer/extracted/processed data/parameters/'
               'GFP-HDEL mv 50_Airyscan Processing_param.mat', 'r') as f:
    P = f['parameters']; E = f['experiment']
    pc   = dict(nScales=float(P['enhance_PC_nScales'][0,0]),
                nOrient=float(P['enhance_PC_nOrient'][0,0]),
                minWavelength=float(P['enhance_PC_minWaveLength'][0,0]),
                noiseMethod=float(P['enhance_PC_noiseMethod'][0,0]))
    skel = dict(thresh_min=float(P['skeleton_thresh_min'][0,0]),
                thresh_max=float(P['skeleton_thresh_max'][0,0]),
                hmin=float(P['skeleton_hmin'][0,0]),
                method=chars(P['skeleton_method'][()]))
    bg   = dict(opening_size=float(P['background_opening_size'][0,0]),
                sigma=float(P['background_sigma'][0,0]))
    micron_per_pixel = float(E['micron_per_pixel'][0,0])
print('phase congruency:', pc)
print('skeleton:', skel)
print('background:', bg)
print('micron per pixel:', micron_per_pixel)

## 5. Load the confocal frame and the manual ground truth

The ground truth was digitized on the **first frame** of the 5-frame stack (`GFP-HDEL mv 50_Airyscan Processing.czi`); we verify this below by scoring every frame and confirming frame 1 agrees best — the annotation frame is identified from the data, not assumed. In the GT image the network skeleton is **0** and background is **255**. Japanese: CZI各フレームと手描きGTを照合し、GTが第1フレームに対応することをデータから確認します。

In [ ]:
import aicspylibczi
from PIL import Image

czi = aicspylibczi.CziFile('/content/analyzer/extracted/GFP-HDEL mv 50_Airyscan Processing.czi')
print('CZI dims:', czi.dims, czi.get_dims_shape())
img0, _ = czi.read_image(C=0, T=0, Z=0)
img0 = np.squeeze(img0).astype(float)
gt = np.array(Image.open('/content/analyzer/extracted/GFP-HDEL mv 50_Airyscan GT.tif')) == 0
print('frame shape', img0.shape, 'dtype(float) | GT skeleton pixels:', int(gt.sum()))

## 6. Input preview — original confocal frame with the reference annotation

Show the actual GFP-HDEL cortical-ER image (frame 1) next to the authors' manually digitized ground-truth skeleton. This is the reference annotation supplied by the authors, not our prediction. Japanese: 元画像(GFP-HDEL 皮質ER)と著者の手描きGT骨格を並べて表示します。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5), constrained_layout=True)
ax[0].imshow(img0, cmap='gray')
ax[0].set_title('Input: GFP-HDEL mv 50 frame 1 (Airyscan)')
ax[1].imshow(img0, cmap='gray')
ax[1].contour(gt, levels=[0.5], colors='cyan', linewidths=0.5)
ax[1].set_title('Input + authors\' manual ground truth (cyan)')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.savefig('/content/input_gt_preview.png', dpi=110)
plt.show()

## 7. AnalyzER-style skeleton extraction

Re-implement the documented pipeline on the annotated frame:
1. **Background subtraction** — the authors subtract an opening-filtered, Gaussian-smoothed background (`background_opening_size=7`, `background_sigma=7`), then normalize.
2. **Phase congruency** — intensity-independent ridge enhancement via `phasepack.phasecongmono` (Kovesi's algorithm; AnalyzER manual §2) with the authors' settings `nScales=4, nOrient=6, minWaveLength=5`, noise method −1, and the algorithm defaults `mult=2.1, sigmaOnf=0.55`.
3. **Hysteresis thresholding + Zhang–Sun skeletonisation** — the authors' method string is `"hyst + h-min"` with `skeleton_thresh_min=0.3`, `skeleton_thresh_max=0.5`; we apply `skimage.filters.apply_hysteresis_threshold` (low 0.3, high 0.5 on max-normalized PC) and `skimage.morphology.skeletonize` (Zhang–Suen). The optional h-minima pre-filter did not improve agreement in our checks and is omitted (documented adaptation).

Japanese: 背景除去→位相一致度(Kovesi)→ヒステリシスしきい値→Zhang–Suen骨格化、という著者記載のパイプラインを著者パラメータのまま再実装します。

In [ ]:
from phasepack import phasecongmono
from skimage.morphology import skeletonize
from skimage.filters import apply_hysteresis_threshold, gaussian
from scipy import ndimage as ndi

im = img0 / img0.max()
bg_img = gaussian(ndi.grey_opening(im, size=int(bg['opening_size'])), sigma=bg['sigma'])
im2 = np.clip(im - bg_img, 0, None)
im2 = im2 / im2.max()

pc_map = phasecongmono(im2, nscale=int(pc['nScales']), minWaveLength=pc['minWavelength'],
                   mult=2.1, sigmaOnf=0.55, noiseMethod=int(pc['noiseMethod']))[0]
pc_map = pc_map / pc_map.max()
skel = skeletonize(apply_hysteresis_threshold(pc_map, skel['thresh_min'], skel['thresh_max']))
print('skeleton pixels:', int(skel.sum()))

## 8. Phase-congruency map

Display the enhanced ridge map — tubules appear as high phase-congruency ridges independent of absolute brightness (the property the paper exploits for intensity-independent segmentation). Japanese: 明るさに依存しない ridge 強調結果を表示します。

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
ax.imshow(pc_map, cmap='magma')
ax.set_title('Phase congruency (nScales=4, nOrient=6, minWavelength=5)')
ax.set_xticks([]); ax.set_yticks([])
plt.savefig('/content/phase_congruency.png', dpi=110)
plt.show()

## 9. Ground-truth comparison — precision / recall / F1

Reproduce AnalyzER's ROC counting (manual §6.14): a predicted skeleton pixel is a **true positive** if a ground-truth pixel lies within a *tolerance* radius, else a **false positive**; unmatched GT pixels are **false negatives**. The manual's typical tolerance is ~half the tubule width; with the authors' sample width ≈ 0.33 µm (width_sample_mean) at 0.0461 µm/px ≈ 7 px, half-width ≈ 3–4 px, so we report a tolerance table from 2–7 px. Japanese: 許容半径つき TP/FN/FP を数え、precision/recall/F1 を許容距離ごとに表にします。

In [ ]:
import pandas as pd
dist_gt = ndi.distance_transform_edt(~gt)
rows = []
for tol in [2, 3, 4, 5, 7]:
    tp = int((skel & (dist_gt <= tol)).sum())
    fp = int((skel & (dist_gt > tol)).sum())
    ds = ndi.distance_transform_edt(~skel)
    fn = int((gt & (ds > tol)).sum())
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    rows.append(dict(tolerance_px=tol, TP=tp, FN=fn, FP=fp,
                     precision=round(p, 3), recall=round(r, 3),
                     **{'F1 (Dice)': round(2 * p * r / max(p + r, 1e-9), 3)}))
pr = pd.DataFrame(rows)
pr.to_csv('/content/precision_recall.csv', index=False)
print(pr.to_string(index=False))

## 10. Frame identity check

Confirm from the data that the ground truth annotates frame 1: score every frame of the stack at tolerance 3 px. Frame 1 should clearly dominate. Japanese: GTが第1フレームに対応することを全フレームのスコアで確認します。

In [ ]:
for T in range(5):
    imT, _ = czi.read_image(C=0, T=T, Z=0); imT = np.squeeze(imT).astype(float)
    x = imT / imT.max()
    bgT = gaussian(ndi.grey_opening(x, size=int(bg['opening_size'])), sigma=bg['sigma'])
    x = np.clip(x - bgT, 0, None); x = x / x.max()
    pcT = phasecongmono(x, nscale=int(pc['nScales']), minWaveLength=pc['minWavelength'],
                        mult=2.1, sigmaOnf=0.55, noiseMethod=int(pc['noiseMethod']))[0]
    pcT = pcT / pcT.max()
    skT = skeletonize(apply_hysteresis_threshold(pcT, skel['thresh_min'], skel['thresh_max']))
    tp = int((skT & (dist_gt <= 3)).sum())
    fp = int((skT & (dist_gt > 3)).sum())
    dsT = ndi.distance_transform_edt(~skT)
    fn = int((gt & (dsT > 3)).sum())
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    print(f'frame {T+1}: F1@3px = {2*p*r/max(p+r,1e-9):.3f}')

## 11. Color-coded comparison overlay

Render the skeleton comparison in the manual's convention (Fig. 6.16): **green** = true positives, **blue** = false positives, **red** = false negatives (tolerance 3 px). Japanese: TP=緑、FP=青、FN=赤の色分けオーバーレイを作成します。

In [ ]:
ds = ndi.distance_transform_edt(~skel)
tp = skel & (dist_gt <= 3)
fp = skel & (dist_gt > 3)
fn = gt & (ds > 3)
rgb = np.full((*img0.shape, 3), 0.08)
for m, c in [(tp, (0.1, 0.9, 0.1)), (fp, (0.2, 0.4, 1.0)), (fn, (1.0, 0.15, 0.15))]:
    rgb[m] = c
fig, ax = plt.subplots(figsize=(6.5, 6.5), constrained_layout=True)
ax.imshow(rgb)
ax.set_title('TP (green) / FP (blue) / FN (red), tolerance 3 px')
ax.set_xticks([]); ax.set_yticks([])
plt.savefig('/content/pr_overlay.png', dpi=110)
plt.show()

## 12. Interpretation and limitations

**Result (validated on this run's execution — see the saved outputs).** With the authors' own parameters, the Python port reaches **F1 ≈ 0.68 at the manual's typical 3 px tolerance** (≈0.79 at 5 px, ≈0.84 at 7 px). The paper's full GUI pipeline reports **F1 = 0.94 ± 0.019 (n=5)** — measured, per the paper, on *central network regions only*, after convex-hull contraction masking of peripheral errors, with per-image parameter optimization and manual editing options that this minimal port does not use.

**What differs (documented adaptations):** no crop/ROI or peripheral masking, no h-minima pre-filter, no per-image threshold re-optimization, single annotated frame instead of n=5 images. The phase-congruency stage (phasepack = Kovesi's own code) and hysteresis + Zhang–Suen skeleton follow the manual and the authors' `param.mat` directly.

**Not reproduced:** cisterna segmentation, graph conversion, width/dynamics/persistency metrics, multivariate statistics.

**References:** Stephens et al., Nat Commun 10:984 (2019) · AnalyzER v1 manual (ORA uuid:cb0e2845-2a9c-495a-84f0-4dd2c5164463) · Kovesi, phase congruency (phasepack PyPI 1.5) · data: ORA "Processing example with ground truth", ODC-BY.

Japanese: 著者のパラメータを用いたPython移植は F1≈0.68(3 px)。論文の0.94は中心領域のみ・マスク・個別最適化を含む完全パイプラインの値であり、単純化の差を説明します。